In [ ]:
%run ../_init_notebook.py

# BTC Direction Research

Análisis estadístico de la dirección del precio de BTC-USD.

**Pregunta central:** ¿Con qué frecuencia el precio está más alto que N días atrás?

Este notebook sienta las bases estadísticas para luego construir modelos predictivos.

## 1. Carga de datos

In [ ]:
btc_path = os.path.join(PROJECT_ROOT, 'data', 'yahoo_1d', 'cripto', 'BTC-USD.csv')

df = pd.read_csv(btc_path, header=[0, 1], index_col=0, parse_dates=True)
# Aplanar columnas multiindex
df.columns = [col[0] for col in df.columns]
df = df.sort_index()

print(f"Rango de datos: {df.index[0].date()} → {df.index[-1].date()}")
print(f"Total de velas: {len(df)}")
df.head()

## 2. ¿Cuántas veces el precio estuvo más alto que hace 3 días?

In [ ]:
LAG = 3  # días atrás para comparar

df['close_lag3'] = df['Close'].shift(LAG)
df['higher_than_lag3'] = (df['Close'] > df['close_lag3']).astype(int)

valid = df.dropna(subset=['close_lag3'])

total = len(valid)
up_count = valid['higher_than_lag3'].sum()
down_count = total - up_count
up_pct = up_count / total * 100

print(f"Total de días analizados: {total}")
print(f"Días donde close > close hace {LAG} días: {up_count} ({up_pct:.1f}%)")
print(f"Días donde close <= close hace {LAG} días: {down_count} ({100-up_pct:.1f}%)")

## 3. Desglose por año

In [ ]:
valid = valid.copy()
valid['year'] = valid.index.year

by_year = valid.groupby('year')['higher_than_lag3'].agg(['sum', 'count'])
by_year.columns = ['up_days', 'total_days']
by_year['up_pct'] = (by_year['up_days'] / by_year['total_days'] * 100).round(1)
by_year['down_pct'] = (100 - by_year['up_pct']).round(1)

print(f"Frecuencia de close > close hace {LAG} días, por año:\n")
display(by_year)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#2ecc71' if p >= 50 else '#e74c3c' for p in by_year['up_pct']]
ax.bar(by_year.index, by_year['up_pct'], color=colors)
ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='50%')
ax.axhline(valid['higher_than_lag3'].mean() * 100, color='orange', linestyle='--', linewidth=1.5, label=f'Media global ({up_pct:.1f}%)')
ax.set_xlabel('Año')
ax.set_ylabel('% días alcistas (vs hace 3d)')
ax.set_title(f'BTC-USD: % días donde Close > Close[-{LAG}d], por año')
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 4. Desglose por mes del año

In [ ]:
valid['month'] = valid.index.month
month_names = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

by_month = valid.groupby('month')['higher_than_lag3'].agg(['sum', 'count'])
by_month.columns = ['up_days', 'total_days']
by_month['up_pct'] = (by_month['up_days'] / by_month['total_days'] * 100).round(1)
by_month.index = month_names

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#2ecc71' if p >= 50 else '#e74c3c' for p in by_month['up_pct']]
ax.bar(by_month.index, by_month['up_pct'], color=colors)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.axhline(up_pct, color='orange', linestyle='--', linewidth=1.5, label=f'Media global ({up_pct:.1f}%)')
ax.set_ylabel('% días alcistas (vs hace 3d)')
ax.set_title(f'BTC-USD: % días donde Close > Close[-{LAG}d], por mes')
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

display(by_month)

## 5. Desglose por día de la semana

In [ ]:
valid['weekday'] = valid.index.dayofweek
day_names = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

by_dow = valid.groupby('weekday')['higher_than_lag3'].agg(['sum', 'count'])
by_dow.columns = ['up_days', 'total_days']
by_dow['up_pct'] = (by_dow['up_days'] / by_dow['total_days'] * 100).round(1)
by_dow.index = [day_names[i] for i in by_dow.index]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if p >= 50 else '#e74c3c' for p in by_dow['up_pct']]
ax.bar(by_dow.index, by_dow['up_pct'], color=colors)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.axhline(up_pct, color='orange', linestyle='--', linewidth=1.5, label=f'Media global ({up_pct:.1f}%)')
ax.set_ylabel('% días alcistas (vs hace 3d)')
ax.set_title(f'BTC-USD: % días donde Close > Close[-{LAG}d], por día de semana')
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

display(by_dow)

## 6. Comparación entre distintos lags (1d, 3d, 5d, 7d, 14d, 30d)

In [ ]:
lags = [1, 2, 3, 5, 7, 10, 14, 21, 30]
results = []

for lag in lags:
    close_lagged = df['Close'].shift(lag)
    higher = (df['Close'] > close_lagged).dropna()
    valid_lag = higher[close_lagged.notna()]
    pct = valid_lag.mean() * 100
    results.append({'lag': lag, 'up_pct': round(pct, 2), 'n': len(valid_lag)})

lag_df = pd.DataFrame(results).set_index('lag')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lag_df.index, lag_df['up_pct'], marker='o', color='#3498db', linewidth=2)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.fill_between(lag_df.index, 50, lag_df['up_pct'], where=lag_df['up_pct'] >= 50, alpha=0.2, color='green')
ax.fill_between(lag_df.index, 50, lag_df['up_pct'], where=lag_df['up_pct'] < 50, alpha=0.2, color='red')
ax.set_xlabel('Lag (días)')
ax.set_ylabel('% días donde Close > Close[-N]')
ax.set_title('BTC-USD: Frecuencia alcista vs distintos lags')
ax.set_xticks(lag_df.index)
plt.tight_layout()
plt.show()

display(lag_df)

## 7. Análisis de rachas (streaks)

¿Cuántos días consecutivos suele estar BTC por encima del precio de hace 3 días?

In [ ]:
signal = df['higher_than_lag3'].dropna().astype(int)

# Calcular rachas
streaks = []
current_val = signal.iloc[0]
current_len = 1

for val in signal.iloc[1:]:
    if val == current_val:
        current_len += 1
    else:
        streaks.append({'direction': 'up' if current_val == 1 else 'down', 'length': current_len})
        current_val = val
        current_len = 1
streaks.append({'direction': 'up' if current_val == 1 else 'down', 'length': current_len})

streaks_df = pd.DataFrame(streaks)

up_streaks = streaks_df[streaks_df['direction'] == 'up']['length']
down_streaks = streaks_df[streaks_df['direction'] == 'down']['length']

print("Rachas ALCISTAS (días consecutivos > lag3):")
print(f"  Promedio: {up_streaks.mean():.1f} días | Mediana: {up_streaks.median():.0f} | Max: {up_streaks.max()}")
print()
print("Rachas BAJISTAS (días consecutivos <= lag3):")
print(f"  Promedio: {down_streaks.mean():.1f} días | Mediana: {down_streaks.median():.0f} | Max: {down_streaks.max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

up_counts = up_streaks.value_counts().sort_index()
down_counts = down_streaks.value_counts().sort_index()

axes[0].bar(up_counts.index, up_counts.values, color='#2ecc71')
axes[0].set_title(f'Distribución de rachas ALCISTAS (vs lag {LAG}d)')
axes[0].set_xlabel('Días consecutivos')
axes[0].set_ylabel('Frecuencia')

axes[1].bar(down_counts.index, down_counts.values, color='#e74c3c')
axes[1].set_title(f'Distribución de rachas BAJISTAS (vs lag {LAG}d)')
axes[1].set_xlabel('Días consecutivos')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 8. Evolución temporal de la frecuencia (rolling 365 días)

In [ ]:
rolling_pct = df['higher_than_lag3'].rolling(365).mean() * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df.index, df['Close'], color='#3498db', linewidth=1)
axes[0].set_ylabel('BTC-USD Close')
axes[0].set_title('Precio BTC-USD')
axes[0].set_yscale('log')

axes[1].plot(rolling_pct.index, rolling_pct, color='#9b59b6', linewidth=1.5)
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1)
axes[1].fill_between(rolling_pct.index, 50, rolling_pct, where=rolling_pct >= 50, alpha=0.2, color='green')
axes[1].fill_between(rolling_pct.index, 50, rolling_pct, where=rolling_pct < 50, alpha=0.2, color='red')
axes[1].set_ylabel('% días alcistas (rolling 365d)')
axes[1].set_title(f'Frecuencia rolling (365d): Close > Close[-{LAG}d]')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()